# Domain C: Contextual Adaptation and History Dependence

This notebook addresses three questions:
- **C1:** Do responses differ between the first and second presentation of the same block type (context-dependent adaptation)?
- **C2:** Does the context (contrast-varying vs. speed-varying) alter tuning curves?
- **C3:** Do responses change across days (multi-day representational drift)?

**Data:** Zarr multimodal stores with ΔF/F calcium traces (X matrix, cells × trials), stimulus metadata (var), cell-type labels & gene expression (obs).
Context structure: Blocks 0,2 = contrast-context (TF fixed at 1 Hz); Blocks 1,3 = speed-context (contrast fixed at 0.8).

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, wilcoxon, mannwhitneyu, spearmanr, pearsonr, ttest_rel
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import get_block_responses, compute_adaptation_index, get_trial_position_responses, exp_decay, linear_CKA
from functions.tuning import naka_rushton, normalization_model

sns.set_context('talk')
sns.set_style('whitegrid')

ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

print(f"Total cells: {len(obs)}, Total trials: {X.shape[1]}")

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

# ── Subclass setup ──
SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]
short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

## C3: Multi-Day Representational Drift

Track how responses to identical stimuli change across the 3 recording days. Compute stability metrics per cell type. Analyze population-level drift via PCA.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C3.1  Cross-day response stability analysis
# ══════════════════════════════════════════════════════════════════════

# The var dataframe has a 'day' column (1, 2, 3) for the 3 recording days
days = sorted(var['day'].unique())
print(f"Days in dataset: {days}")

# Compute mean response per cell per stimulus condition per day
# Use all blocks combined; conditions = orientation × contrast × TF (use stim_index_block)
# Simplify: use orientation tuning vector per day (contrast-context blocks, collapse contrast)

day_ori_responses = {}
for day in days:
    day_mask = var['day'] == day
    contrast_day = day_mask & var['stim_block'].isin([0.0, 2.0])
    resp = np.zeros((adata.n_obs, len(orientations)))
    for i, ori in enumerate(orientations):
        mask = contrast_day & (var['orientation'] == ori)
        trial_idx = np.where(mask.values)[0]
        if len(trial_idx) > 0:
            resp[:, i] = np.nanmean(X[:, trial_idx], axis=1)
    day_ori_responses[day] = resp

# ── Cross-day correlation per cell ──
from itertools import combinations

day_pairs = list(combinations(days, 2))
stability_records = []

for cell_i in range(adata.n_obs):
    for d1, d2 in day_pairs:
        v1 = day_ori_responses[d1][cell_i]
        v2 = day_ori_responses[d2][cell_i]
        if np.std(v1) > 1e-6 and np.std(v2) > 1e-6:
            r, _ = pearsonr(v1, v2)
        else:
            r = np.nan
        stability_records.append({
            'cell': cell_i,
            'day_pair': f'{d1}-{d2}',
            'correlation': r,
            'subclass': obs['subclass_name'].iloc[cell_i],
            'subclass_short': SUBCLASS_SHORT.get(obs['subclass_name'].iloc[cell_i], 'Other'),
        })

stab_df = pd.DataFrame(stability_records)

# ── Visualization: Cross-day stability by subclass ──
fig, axes = plt.subplots(1, len(day_pairs), figsize=(6*len(day_pairs), 5))
if len(day_pairs) == 1:
    axes = [axes]

for ax, dp in zip(axes, [f'{d1}-{d2}' for d1, d2 in day_pairs]):
    sub = stab_df[stab_df['day_pair'] == dp]
    sns.violinplot(data=sub[sub['subclass'].isin(present_subclasses)],
                   x='subclass_short', y='correlation', order=short_order,
                   palette=short_pal, cut=0, inner='box', ax=ax)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    ax.set_title(f'C3: Day {dp} Stability', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Pearson r (orientation vector)')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('C3: Cross-Day Tuning Stability by Subclass', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Kruskal-Wallis across subclasses for stability ──
mean_stab = stab_df.groupby(['cell', 'subclass'])['correlation'].mean().reset_index()
groups = [mean_stab.loc[mean_stab['subclass'] == s, 'correlation'].dropna().values
          for s in present_subclasses]
groups = [g for g in groups if len(g) >= 3]
if len(groups) >= 2:
    stat, p = kruskal(*groups)
    print(f"Cross-day stability Kruskal-Wallis: H={stat:.2f}, p={p:.2e}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# C3.2  Population-level drift via PCA
# ══════════════════════════════════════════════════════════════════════
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Build population response matrix per day: cells × orientations
# Then apply PCA to the combined space
all_day_resp = np.vstack([day_ori_responses[d] for d in days])  # (3*n_cells) × 8
day_labels = np.concatenate([[d]*adata.n_obs for d in days])

pca = PCA(n_components=3)
pca_coords = pca.fit_transform(all_day_resp)

fig = plt.figure(figsize=(14, 5))

# 2D PCA colored by day
ax1 = fig.add_subplot(131)
for d in days:
    mask = day_labels == d
    ax1.scatter(pca_coords[mask, 0], pca_coords[mask, 1], alpha=0.3, s=10, label=f'Day {d}')
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax1.legend()
ax1.set_title('C3: Population PCA by Day')

# Track mean position per day per subclass
ax2 = fig.add_subplot(132)
for sc in present_subclasses:
    sc_mask_base = obs['subclass_name'].values == sc
    centroids = []
    for d in days:
        d_mask = day_labels == d
        combined_mask = np.zeros(len(day_labels), dtype=bool)
        # This subclass in this day
        start = (d - days[0]) * adata.n_obs
        end = start + adata.n_obs
        combined_mask[start:end] = sc_mask_base
        combined_mask = combined_mask & d_mask
        if combined_mask.sum() > 0:
            centroids.append(pca_coords[combined_mask].mean(axis=0))
    if len(centroids) == len(days):
        centroids = np.array(centroids)
        ax2.plot(centroids[:, 0], centroids[:, 1], 'o-', color=SUBCLASS_COLORS[sc],
                label=SUBCLASS_SHORT[sc], linewidth=2, markersize=8)
        # Arrow from day 1 to day 3
        ax2.annotate('', xy=centroids[-1, :2], xytext=centroids[0, :2],
                    arrowprops=dict(arrowstyle='->', color=SUBCLASS_COLORS[sc], lw=1.5))
ax2.set_xlabel(f'PC1')
ax2.set_ylabel(f'PC2')
ax2.legend(fontsize=7)
ax2.set_title('C3: Subclass Centroid Drift')

# Euclidean drift distance per subclass
ax3 = fig.add_subplot(133)
drift_data = []
for sc in present_subclasses:
    sc_mask_base = obs['subclass_name'].values == sc
    for cell_i in np.where(sc_mask_base)[0]:
        coords = []
        for d in days:
            idx = (d - days[0]) * adata.n_obs + cell_i
            coords.append(pca_coords[idx])
        coords = np.array(coords)
        # Total drift = sum of day-to-day euclidean distances
        total_drift = sum(np.linalg.norm(coords[j+1] - coords[j]) for j in range(len(days)-1))
        drift_data.append({'subclass_short': SUBCLASS_SHORT[sc], 'drift': total_drift})

drift_df = pd.DataFrame(drift_data)
sns.boxplot(data=drift_df, x='subclass_short', y='drift', order=short_order,
            palette=short_pal, ax=ax3)
ax3.set_title('C3: Total PCA Drift by Subclass')
ax3.set_xlabel('')
ax3.set_ylabel('Total PCA drift (Days 1→3)')
ax3.tick_params(axis='x', rotation=45)

plt.suptitle('C3: Multi-Day Representational Drift', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── CKA: compare population RDMs across days ──
print("\n=== CKA between days (population orientation responses) ===")
for d1, d2 in day_pairs:
    cka = linear_CKA(day_ori_responses[d1], day_ori_responses[d2])
    print(f"  Day {d1} vs Day {d2}: CKA = {cka:.4f}")